In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split


### 1. Загрузите выборку из файла gbm-data.csv с помощью pandas и преобразуйте ее в массив numpy (параметр values у датафрейма). В первой колонке файла с данными записано, была или нет реакция. Все остальные колонки (d1-d1776) содержат различные характеристики молекулы. Разбейте выборку на обучающую и тестовую, используя функцию train_test_split с параметрами test_size = 0.8 и random_state = 241.


In [2]:
data = pd.read_csv('gbm-data.csv').values
y = data[:, 0]
X = data[:, 1:]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8, random_state=241)


### 2. Обучите GradientBoostingClassifier с параметрами n_estimators=250, verbose=True, random_state=241 и для каждого значения learning_rate из списка [1, 0.5, 0.3, 0.2, 0.1] проделайте следующее: используйте метод staged_decision_function для предсказания качества на обучающей и тестовой выборке на каждой итерации; преобразуйте полученное предсказание по формуле 1 / (1 + e^(-y_pred)), где y_pred — предсказанное значение; вычислите и постройте график значений log-loss на обучающей и тестовой выборках, а также найдите минимальное значение метрики и номер итерации, на которой оно достигается.


In [3]:
def sigmoid(y_pred):
    return 1 / (1 + np.exp(-y_pred))

losses = {}
for learning_rate in [1, 0.5, 0.3, 0.2, 0.1]:
    clf = GradientBoostingClassifier(n_estimators=250, learning_rate=learning_rate, random_state=241)
    clf.fit(X_train, y_train)
    train_loss = []
    test_loss = []
    for train_pred, test_pred in zip(clf.staged_decision_function(X_train), clf.staged_decision_function(X_test)):
        train_loss.append(log_loss(y_train, sigmoid(train_pred.ravel())))
        test_loss.append(log_loss(y_test, sigmoid(test_pred.ravel())))
    losses[learning_rate] = {'train': train_loss, 'test': test_loss}

import matplotlib.pyplot as plt
plt.figure()
plt.plot(losses[0.2]['test'], 'r', linewidth=2)
plt.plot(losses[0.2]['train'], 'g', linewidth=2)
plt.legend(['test', 'train'])


### 3. Как можно охарактеризовать график качества на тестовой выборке, начиная с некоторой итерации: переобучение (overfitting) или недообучение (underfitting)? В ответе укажите одно из слов overfitting либо underfitting.


In [4]:
print('overfitting')


overfitting


### 4. Приведите минимальное значение log-loss на тестовой выборке и номер итерации, на котором оно достигается, при learning_rate = 0.2.


In [5]:
test_loss_02 = losses[0.2]['test']
best_iteration = int(np.argmin(test_loss_02)) + 1
best_loss = test_loss_02[best_iteration - 1]
print(round(best_loss, 2), best_iteration)


0.53 37


### 5. На этих же данных обучите RandomForestClassifier с количеством деревьев, равным количеству итераций, на котором достигается наилучшее качество у градиентного бустинга из предыдущего пункта, random_state=241 и остальными параметрами по умолчанию. Какое значение log-loss на тесте получается у этого случайного леса? Не забывайте, что предсказания нужно получать с помощью функции predict_proba.


In [6]:
forest = RandomForestClassifier(n_estimators=best_iteration, random_state=241)
forest.fit(X_train, y_train)
forest_loss = log_loss(y_test, forest.predict_proba(X_test)[:, 1])
print(round(forest_loss, 2))


0.54
